# SaulGPT Fine-Tuning
QLoRA fine-tuning of Llama 3.1 8B on Indian legal data using Unsloth on Google Colab T4.

In [ ]:
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo


## Step 1: Load Base Model

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=1024,
    load_in_4bit=True,
)


## Step 2: Configure LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)


## Step 3: Load Dataset
Primary: Aalap instruction dataset. If gated/unavailable, use the fallback cell below.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("opennyaiorg/aalap_instruction_dataset", split="train")

SYSTEM_PROMPT = "You are SaulGPT, an expert in Indian Law specializing in BNS, IPC, and the Constitution of India."

def format_example(example):
    return {"text": f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{example.get('user_prompt', example.get('input_text', ''))}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{example.get('output_text', '')}<|eot_id|>"}

dataset = dataset.map(format_example)


### Fallback Dataset (if Aalap is unavailable)

In [ ]:
# FALLBACK: Use this cell if Aalap dataset is unavailable
# Comment out Cell 8 and run this cell instead
SYSTEM_PROMPT = "You are SaulGPT, an expert in Indian Law specializing in BNS, IPC, and the Constitution of India."

legal_qa_pairs = [
    {"instruction": "What is the punishment for murder under BNS?", "response": "Under Section 102 of the Bharatiya Nyaya Sanhita (BNS) 2023, the punishment for murder is death or imprisonment for life."},
    {"instruction": "What is Section 302 of IPC?", "response": "Section 302 of the Indian Penal Code (IPC) deals with punishment for murder — death or imprisonment for life, and fine."},
    {"instruction": "What is the right to equality under the Constitution?", "response": "Article 14 of the Constitution of India guarantees the right to equality before law and equal protection of laws within the territory of India."},
    {"instruction": "What is theft under BNS?", "response": "Section 303 of the Bharatiya Nyaya Sanhita (BNS) 2023 defines theft as dishonestly taking movable property out of the possession of any person without consent."},
    {"instruction": "What is the punishment for theft under IPC?", "response": "Section 379 of the IPC prescribes imprisonment up to 3 years, or fine, or both for theft."},
    {"instruction": "What is culpable homicide under BNS?", "response": "Section 100 of the BNS defines culpable homicide as causing death by an act done with intention or knowledge that it is likely to cause death."},
    {"instruction": "What is the right to freedom of speech in India?", "response": "Article 19(1)(a) of the Constitution guarantees freedom of speech and expression, subject to reasonable restrictions under Article 19(2)."},
    {"instruction": "What is assault under BNS?", "response": "Section 131 of the BNS defines assault as making any gesture or preparation intending or knowing it to be likely to cause apprehension of criminal force."},
    {"instruction": "What is the punishment for robbery under IPC?", "response": "Section 392 of the IPC prescribes rigorous imprisonment up to 10 years and fine for robbery."},
    {"instruction": "What is the right against self-incrimination?", "response": "Article 20(3) of the Constitution provides that no person accused of any offence shall be compelled to be a witness against himself."},
]

# Expand to 100+ examples by adding more pairs
import random
expanded_pairs = legal_qa_pairs * 12  # 120 examples
random.shuffle(expanded_pairs)

def format_fallback(example):
    return {"text": f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{example['response']}<|eot_id|>"}

from datasets import Dataset
dataset = Dataset.from_list(expanded_pairs).map(format_fallback)
print(f"Fallback dataset: {len(dataset)} examples")


## Step 4: Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not FastLanguageModel.is_bfloat16_supported(),
        bf16=FastLanguageModel.is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        output_dir="outputs",
    ),
)
trainer.train()


## Step 5: Export to GGUF

In [ ]:
model.save_pretrained_gguf("saulgpt-model", tokenizer, quantization_method="q4_k_m")
print("GGUF saved to saulgpt-model/")


## Step 6: Download GGUF
Download the GGUF file from Colab to your local machine.

In [ ]:
from google.colab import files
import os
gguf_files = [f for f in os.listdir("saulgpt-model") if f.endswith(".gguf")]
print(f"GGUF files: {gguf_files}")
for f in gguf_files:
    files.download(f"saulgpt-model/{f}")


## Step 7: Create Ollama Modelfile

In [ ]:
modelfile_content = '''FROM ./saulgpt-llama-3.1-8b-q4_k_m.gguf

TEMPLATE """{{ if .System }}<|start_header_id|>system<|end_header_id|>

{{ .System }}<|eot_id|>{{ end }}{{ if .Prompt }}<|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|>{{ end }}<|start_header_id|>assistant<|end_header_id|>

{{ .Response }}<|eot_id|>"""

SYSTEM """You are SaulGPT, an expert legal AI assistant specializing in Indian law. You provide precise, cited legal information based on the Bharatiya Nyaya Sanhita (BNS), Indian Penal Code (IPC), and the Constitution of India. Always cite specific sections and acts. You are NOT a lawyer — you provide informational guidance only."""

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER stop "<|start_header_id|>"
PARAMETER stop "<|end_header_id|>"
PARAMETER stop "<|eot_id|>"
'''
with open("Modelfile", "w") as f:
    f.write(modelfile_content)
print("Modelfile written. Copy contents to ml/Modelfile on your local machine.")
print(modelfile_content)
